# Pratica 14 — Progetto integrativo

**Corso:** Statistica Applicata con Python · **Blocco 5:** Chiusura · **Tempo stimato:** 3-4 ore di lavoro autonomo (un caso)
**Lezione associata:** `notebooks_clase/clase14_proyecto_integrador_IT.ipynb` · **Capitolo:** `manual/14_proyecto_integrador_IT.md`

## Come funziona questa pratica

- Ci sono **tre casi**: **A** (salute: diabete), **B** (e-commerce con un test A/B) e **C** (prezzi delle abitazioni in California). **Scegline UNO e completalo**; se ti avanza tempo, fai gli altri.
- Ogni caso ha **3 passi** con una cella `# TODO` e un **controllo automatico** (`assert`): (1) audit ed esplorazione, (2) modello o test adeguato con diagnostica, (3) un dizionario `resultado_X` con le cifre chiave.
- Alla fine, la **sfida finale** ti chiede di scrivere il report di una pagina (`informe_final`). Il controllo verifica solo il minimo (sezioni, cifre e limiti); la qualità la valuti tu con la **rubrica** dell'ultima cella.
- Esegui le celle di **setup e dati** e poi **solo le celle del tuo caso** (quelle degli altri casi daranno errore se non le completi: è normale).
- Le **soluzioni** dei tre casi sono in una sezione a parte in fondo. **Non aprirle prima di aver provato.**
- Ricorda: A e C sono dati **osservazionali** (niente "causa"); in B la causalità è difendibile solo se la randomizzazione regge: per questo si verifica.

---
## Cella 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests
import re
import unicodedata

def vif_de(X):
    """VIF di ogni colonna di X (DataFrame numerico senza costante). Restituisce un dizionario {colonna: VIF}."""
    Xc = sm.add_constant(X.astype(float))
    return {col: variance_inflation_factor(Xc.values, i) for i, col in enumerate(Xc.columns) if col != "const"}

print("Librerie caricate correttamente.")

---
## Dati dei tre casi

Vengono caricati i tre dataset (A e C: CSV locale → `scikit-learn` → sintetico; B: generato con seme fisso).

In [ ]:
def generar_diabetes_sintetico(n=442, semilla=2026):
    """Dataset SINTETICO con le stesse colonne di `diabetes` (esempio illustrativo, NON sono pazienti reali)."""
    r = np.random.default_rng(semilla)
    age = np.round(np.clip(r.normal(48.5, 13, n), 19, 79))
    sex = r.choice([1.0, 2.0], size=n)
    bmi = np.round(np.clip(r.normal(26.4, 4.4, n), 18, 42), 1)
    bp = np.round(np.clip(70 + 1.0 * (bmi - 26) + 0.3 * age + r.normal(0, 11, n), 62, 133))
    s1 = np.round(np.clip(r.normal(189, 34, n), 97, 301))
    s2 = np.round(np.clip(0.75 * (s1 - 189) + 115 + r.normal(0, 6, n), 42, 242), 1)   # molto correlata con s1
    s3 = np.round(np.clip(50 - 0.15 * (s2 - 115) + r.normal(0, 11, n), 22, 99))
    s4 = np.round(np.clip(4.07 + 0.02 * (s1 - 189) - 0.05 * (s3 - 50) + r.normal(0, 0.6, n), 2, 9), 2)
    s5 = np.round(np.clip(4.6 + 0.004 * (s1 - 189) - 0.008 * (s3 - 50) + r.normal(0, 0.3, n), 3.2, 6.1), 4)
    s6 = np.round(np.clip(r.normal(91, 11.5, n), 58, 124))
    target = np.round(np.clip(152 + 5.5 * (bmi - 26.4) + 1.1 * (bp - 94.6) + 40 * (s5 - 4.6) - 1.0 * (s3 - 50)
                              - 20 * (sex - 1.5) + r.normal(0, 55, n), 25, 346))
    return pd.DataFrame({"age": age, "sex": sex, "bmi": bmi, "bp": bp, "s1": s1, "s2": s2, "s3": s3,
                         "s4": s4, "s5": s5, "s6": s6, "target": target})


def cargar_diabetes():
    """Restituisce (DataFrame, origine): CSV locale, altrimenti scikit-learn (valori originali), altrimenti dati sintetici."""
    for ruta in ("../datos/diabetes.csv", "datos/diabetes.csv", "diabetes.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV locale"
    try:
        from sklearn.datasets import load_diabetes
        return load_diabetes(as_frame=True, scaled=False).frame, "scikit-learn"
    except Exception:
        return generar_diabetes_sintetico(), "sintetico"

def generar_california_sintetico(n=5000, semilla=42):
    """Dataset SINTETICO con le stesse colonne di California housing (esempio illustrativo: le relazioni sono inventate)."""
    g = np.random.default_rng(semilla)
    MedInc = np.clip(g.lognormal(1.25, 0.5, n), 0.5, 15)
    HouseAge = g.integers(1, 53, n).astype(float)
    AveRooms = np.clip(3.0 + 0.35 * MedInc + g.normal(0, 0.7, n), 2, 9)
    AveBedrms = np.clip(0.2 * AveRooms + g.normal(0, 0.05, n), 0.6, 2)
    Population = np.round(g.lognormal(7.0, 0.6, n))
    AveOccup = np.clip(g.lognormal(1.05, 0.25, n), 1.2, 5.5)
    Latitude = np.round(g.uniform(32.5, 42.0, n), 2)
    Longitude = np.round(g.uniform(-124.3, -114.5, n), 2)
    valor = (0.5 + 0.42 * MedInc + 0.012 * HouseAge - 0.15 * AveRooms - 0.3 * AveOccup
             + 0.05 * (37 - Latitude) + g.normal(0, 0.6, n))
    MedHouseVal = np.clip(valor, 0.15, 5.00001)
    return pd.DataFrame({"MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms, "AveBedrms": AveBedrms,
                         "Population": Population, "AveOccup": AveOccup, "Latitude": Latitude,
                         "Longitude": Longitude, "MedHouseVal": MedHouseVal})


def cargar_california():
    """Restituisce (DataFrame, origine): CSV locale, altrimenti scikit-learn, altrimenti dati sintetici."""
    for ruta in ("../datos/california_housing.csv", "datos/california_housing.csv", "california_housing.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV locale"
    try:
        from sklearn.datasets import fetch_california_housing
        datos = fetch_california_housing(as_frame=True).frame
        return datos.sample(n=5000, random_state=42).reset_index(drop=True), "scikit-learn"
    except Exception:
        return generar_california_sintetico(), "sintetico"

def generar_ecommerce(n=12000, semilla=1):
    """Genera il caso B: un test A/B di e-commerce con dati SINTETICI (esempio illustrativo).
    Effetto vero di B sulla conversione: positivo e moderato (+0,10 in log-odds su desktop, +0,30 su mobile).
    Trappola deliberata: l'assegnazione NON è 50/50 (su mobile il 54 % va a B) -> SRM e squilibrio dei dispositivi.
    Gli introiti sono 0 se non converte e lognormali (coda lunga) se converte."""
    r = np.random.default_rng(semilla)
    dispositivo = np.where(r.random(n) < 0.55, "movil", "escritorio")
    canal = r.choice(["organico", "pago", "email"], size=n, p=[0.5, 0.3, 0.2])
    sesiones_previas = r.poisson(2.0, n)
    prob_b = np.where(dispositivo == "movil", 0.54, 0.50)
    grupo = np.where(r.random(n) < prob_b, "B", "A")
    logit = (-2.7 + np.where(dispositivo == "movil", -0.25, 0.0)
             + np.select([canal == "pago", canal == "email"], [-0.30, 0.45], 0.0)
             + 0.10 * sesiones_previas
             + (grupo == "B") * np.where(dispositivo == "movil", 0.30, 0.10))
    convirtio = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    ingresos = np.where(convirtio == 1, np.round(r.lognormal(3.6, 0.9, n), 2), 0.0)
    return pd.DataFrame({"id_usuario": np.arange(1, n + 1), "grupo": grupo, "dispositivo": dispositivo,
                         "canal": canal, "sesiones_previas": sesiones_previas, "convirtio": convirtio,
                         "ingresos": ingresos})


diab, origen_a = cargar_diabetes()
cali, origen_c = cargar_california()
ecom = generar_ecommerce()
print("Caso A (diabete):", diab.shape, "— origine:", origen_a)
print("Caso B (e-commerce, sintetico):", ecom.shape)
print("Caso C (abitazioni della California):", cali.shape, "— origine:", origen_c)

---
# Caso A — Salute: progressione del diabete

**Contesto.** 442 pazienti con diabete. Per ciascuno: `age`, `sex` (codificato 1/2; il dataset non dice quale sia quale, quindi **non interpretare il segno** come "uomo/donna"), `bmi` (indice di massa corporea), `bp` (pressione arteriosa media), sei esami del sangue `s1`-`s6` e `target`, una misura quantitativa della **progressione della malattia a un anno** (valori originali, non scalati). Sono dati **osservazionali**.

**Domanda di business.** Un'équipe clinica vuole sapere quali caratteristiche di base si associano a una maggiore progressione, per dare priorità al follow-up.

**Rotta.** Audit (lezioni 2-3) → correlazioni e multicollinearità (lezione 12: VIF) → regressione multipla con errori robusti HC3 (lezione 10) → effetto del `bmi` con IC.
**Trappole:** `s1`, `s2` (e `s3`) misurano cose simili, con VIF molto alti; con n = 442 non si può affermare causa né prevedere bene senza validazione.

### Caso A · Passo 1 — Audit ed esplorazione

Calcola su `diab`: `n_a` (righe), `n_faltantes_a` (totale dei valori mancanti), `n_duplicados_a` (righe duplicate), `r_bmi` (correlazione di Pearson tra `bmi` e `target`), `var_max_corr` (nome della variabile, escluso `target`, con la maggiore correlazione **assoluta** con `target`) e `r_s1_s2` (correlazione tra `s1` e `s2`: il segnale di multicollinearità).

In [ ]:
n_a = None              # TODO
n_faltantes_a = None    # TODO
n_duplicados_a = None   # TODO
r_bmi = None            # TODO
var_max_corr = None     # TODO: suggerimento: diab.drop(columns="target").corrwith(...)
r_s1_s2 = None          # TODO

# Controllo automatico
assert all(v is not None for v in (n_a, n_faltantes_a, n_duplicados_a, r_bmi, var_max_corr, r_s1_s2)), "Completa le righe contrassegnate con TODO"
assert n_a == len(diab), "n_a = len(diab)"
assert n_faltantes_a == int(diab.isna().sum().sum()), "somma di tutti i mancanti: diab.isna().sum().sum()"
assert n_duplicados_a == int(diab.duplicated().sum()), "diab.duplicated().sum()"
assert np.isclose(r_bmi, diab["bmi"].corr(diab["target"])), "r di Pearson tra bmi e target"
assert var_max_corr == diab.drop(columns="target").corrwith(diab["target"]).abs().idxmax(), "usa il valore assoluto della correlazione e idxmax()"
assert np.isclose(r_s1_s2, diab["s1"].corr(diab["s2"])), "r di Pearson tra s1 e s2"
print(f"n = {n_a}, mancanti = {n_faltantes_a}, duplicati = {n_duplicados_a}, r(bmi, target) = {r_bmi:.3f}, maggiore |r|: {var_max_corr}, r(s1, s2) = {r_s1_s2:.3f}")
print("Caso A, passo 1 corretto ✔")

### Caso A · Passo 2 — Multicollinearità e modello con errori robusti

Con la matrice `X_a` (tutte le esplicative, `sex` come 0/1) calcola `vif_a` (dizionario di VIF, con la funzione `vif_de` del setup), `vif_max_a` (il VIF maggiore) e `variables_vif_alto` (lista **ordinata** delle variabili con VIF > 10). Poi adatta `modelo_a`: la regressione di `target` su `age`, `sex` (categorica), `bmi`, `bp`, `s3`, `s5` e `s6` (senza `s1`, `s2` e `s4`, che sono ridondanti) con **errori standard robusti HC3**.

In [ ]:
X_a = diab.drop(columns="target").copy()
X_a["sex"] = (X_a["sex"] == 2).astype(float)     # 0/1: solo per poter calcolare il VIF

vif_a = None                # TODO
vif_max_a = None            # TODO
variables_vif_alto = None   # TODO
modelo_a = None             # TODO: smf.ols("target ~ age + C(sex) + ...", data=diab).fit(cov_type="HC3")

# Controllo automatico
assert all(v is not None for v in (vif_a, vif_max_a, variables_vif_alto, modelo_a)), "Completa le righe contrassegnate con TODO"
_Xn = X_a.astype(float).values
_ref = dict(zip(X_a.columns, np.diag(np.linalg.inv(np.corrcoef(_Xn.T)))))      # VIF = diagonale dell'inversa della matrice di correlazione
assert set(vif_a) == set(X_a.columns) and all(np.isclose(vif_a[c], _ref[c], rtol=1e-6) for c in _ref), "vif_a = vif_de(X_a): un VIF per ogni colonna di X_a"
assert np.isclose(vif_max_a, max(_ref.values())), "vif_max_a = max(vif_a.values())"
assert variables_vif_alto == sorted(c for c, v in _ref.items() if v > 10), "sorted([c for c, v in vif_a.items() if v > 10])"
assert getattr(modelo_a, "cov_type", None) == "HC3", "manca .fit(cov_type='HC3')"
assert set(modelo_a.params.index) == {"Intercept", "C(sex)[T.2.0]", "age", "bmi", "bp", "s3", "s5", "s6"}, "formula: target ~ age + C(sex) + bmi + bp + s3 + s5 + s6"
print(f"VIF massimo (modello completo) = {vif_max_a:.1f}; variabili con VIF > 10:", variables_vif_alto)
print("Coefficiente di bmi (HC3):", round(modelo_a.params["bmi"], 3))
print("Caso A, passo 2 corretto ✔")

### Caso A · Passo 3 — Risultato con incertezza

Con `modelo_a` costruisci il dizionario `resultado_a` con le chiavi: `n` (osservazioni del modello), `coef_bmi` (coefficiente di `bmi`: variazione attesa di `target` per ogni unità di `bmi`, a parità del resto), `ic_inf` e `ic_sup` (IC 95 % con HC3), `r2_adj` (R² corretto), `vif_max` (VIF maggiore **del modello ridotto**; calcolalo con `vif_de` su `age`, `sex`, `bmi`, `bp`, `s3`, `s5`, `s6`) e `causal` (`True` o `False`: questi dati autorizzano ad affermare che ridurre il `bmi` ridurrà la progressione?).

In [ ]:
X_red = X_a[["age", "sex", "bmi", "bp", "s3", "s5", "s6"]]

resultado_a = {
    "n": None,          # TODO
    "coef_bmi": None,   # TODO
    "ic_inf": None,     # TODO
    "ic_sup": None,     # TODO
    "r2_adj": None,     # TODO
    "vif_max": None,    # TODO
    "causal": None,     # TODO
}

# Controllo automatico
assert all(v is not None for v in resultado_a.values()), "Completa le righe contrassegnate con TODO"
_r = resultado_a
assert _r["n"] == len(diab), "n = numero di osservazioni del modello (modelo_a.nobs)"
assert np.isclose(_r["coef_bmi"], modelo_a.params["bmi"]), "coef_bmi = modelo_a.params['bmi']"
_ic = modelo_a.conf_int().loc["bmi"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 %: modelo_a.conf_int().loc['bmi']"
assert _r["ic_inf"] < _r["coef_bmi"] < _r["ic_sup"], "l'IC deve contenere la stima"
assert np.isclose(_r["r2_adj"], smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit().rsquared_adj), "R² corretto: .rsquared_adj (quello del modello senza HC3 è lo stesso valore)"
_v = np.diag(np.linalg.inv(np.corrcoef(X_red.astype(float).values.T))).max()
assert np.isclose(_r["vif_max"], _v, rtol=1e-6), "vif_max = max(vif_de(X_red).values())"
assert _r["causal"] is False, "sono dati osservazionali: non si può affermare una causa"
print(f"Effetto di bmi: {_r['coef_bmi']:.2f} per unità (IC 95 % HC3: {_r['ic_inf']:.2f} a {_r['ic_sup']:.2f}); R² corretto = {_r['r2_adj']:.3f}; VIF massimo = {_r['vif_max']:.2f}")
print("Caso A, passo 3 corretto ✔")

---
# Caso B — E-commerce: la nuova pagina di pagamento funziona?

**Contesto.** Dati **sintetici** (*esempio illustrativo*) di un test A/B con ~12 000 utenti: `id_usuario`, `grupo` (`A` controllo, `B` nuova versione), `dispositivo` (`movil`/`escritorio`), `canal` (`organico`/`pago`/`email`), `sesiones_previas`, `convirtio` (0/1) e `ingresos` (0 se non converte; con coda lunga se converte). La funzione `generar_ecommerce` è documentata: puoi leggerla.

**Domanda di business.** L'azienda deve lanciare la versione B a tutti gli utenti? E in quali segmenti?

**Rotta.** Audit (lezioni 2-3) → **controlli di salute** dell'esperimento: SRM ed equilibrio dei dispositivi (lezione 13) → test per due proporzioni con IC della differenza (lezioni 7 e 13) → segmenti dispositivo × canale con **correzione per i confronti multipli** (lezione 9).
**Trappole:** la ripartizione **non** è 50/50 (SRM) e sospetta da dove viene; le medie di `ingresos` sono dominate dalla coda lunga; sei segmenti sono sei contrasti.

### Caso B · Passo 1 — Audit ed esplorazione

Su `ecom` calcola: `n_b` (righe), `n_faltantes_b` (totale dei mancanti), `n_duplicados_id` (identificatori `id_usuario` ripetuti), `tasa_A` e `tasa_B` (tasso di conversione di ciascun gruppo, **in %**) e `asimetria_ingresos` (asimmetria, `scipy.stats.skew`, di `ingresos` **solo tra chi converte**: perché la media semplice non serve?).

In [ ]:
n_b = None                 # TODO
n_faltantes_b = None       # TODO
n_duplicados_id = None     # TODO
tasa_A = None              # TODO: in %, non in proporzione
tasa_B = None              # TODO
asimetria_ingresos = None  # TODO

# Controllo automatico
assert all(v is not None for v in (n_b, n_faltantes_b, n_duplicados_id, tasa_A, tasa_B, asimetria_ingresos)), "Completa le righe contrassegnate con TODO"
assert n_b == len(ecom), "n_b = len(ecom)"
assert n_faltantes_b == int(ecom.isna().sum().sum()) and n_duplicados_id == int(ecom["id_usuario"].duplicated().sum()), "ecom.isna().sum().sum() e ecom['id_usuario'].duplicated().sum()"
assert np.isclose(tasa_A, ecom.loc[ecom["grupo"] == "A", "convirtio"].mean() * 100), "tasa_A: media di convirtio nel gruppo A, moltiplicata per 100"
assert np.isclose(tasa_B, ecom.loc[ecom["grupo"] == "B", "convirtio"].mean() * 100), "tasa_B: media di convirtio nel gruppo B, moltiplicata per 100"
assert np.isclose(asimetria_ingresos, stats.skew(ecom.loc[ecom["convirtio"] == 1, "ingresos"])), "stats.skew di ingresos filtrando convirtio == 1"
assert asimetria_ingresos > 1, "un'asimmetria > 1 indica coda lunga: meglio la mediana o il bootstrap della media"
print(f"n = {n_b}, mancanti = {n_faltantes_b}, id ripetuti = {n_duplicados_id}, conversione A = {tasa_A:.2f} %, conversione B = {tasa_B:.2f} %, asimmetria degli introiti = {asimetria_ingresos:.2f}")
print("Caso B, passo 1 corretto ✔")

### Caso B · Passo 2 — Controlli di salute e test per due proporzioni

Calcola: `tamanos_b` (dizionario con gli utenti di `A` e di `B`), `chi2_srm` e `p_srm` (chi-quadro di bontà di adattamento contro una ripartizione 50/50, con `scipy.stats.chisquare`), `srm_detectado` (`True` se `p_srm < 0.001`), `p_dispositivos` (p-value del chi-quadro di indipendenza tra `dispositivo` e `grupo`, con `chi2_contingency`), e il test per due proporzioni: `dif_pp` (tasso di B meno tasso di A, in punti percentuali), `ic_inf_pp` e `ic_sup_pp` (IC 95 % della differenza con `confint_proportions_2indep`, metodo `"newcomb"`, B per primo, in pp) e `p_valor` (`proportions_ztest`, B per primo).

In [ ]:
conv = ecom.groupby("grupo")["convirtio"].agg(["sum", "count"])     # conversioni e utenti per gruppo

tamanos_b = None       # TODO
chi2_srm = None        # TODO
p_srm = None           # TODO
srm_detectado = None   # TODO
p_dispositivos = None  # TODO
dif_pp = None          # TODO
ic_inf_pp = None       # TODO
ic_sup_pp = None       # TODO
p_valor = None         # TODO

# Controllo automatico
assert all(v is not None for v in (tamanos_b, chi2_srm, p_srm, srm_detectado, p_dispositivos, dif_pp, ic_inf_pp, ic_sup_pp, p_valor)), "Completa le righe contrassegnate con TODO"
assert tamanos_b == {"A": int(conv.loc["A", "count"]), "B": int(conv.loc["B", "count"])}, "tamanos_b = {'A': utenti di A, 'B': utenti di B}"
_c2, _p = stats.chisquare([conv.loc["A", "count"], conv.loc["B", "count"]])
assert np.isclose(chi2_srm, _c2) and np.isclose(p_srm, _p), "stats.chisquare([n_A, n_B]) ripartisce per default in parti uguali"
assert srm_detectado == bool(_p < 0.001), "srm_detectado = p_srm < 0.001"
assert np.isclose(p_dispositivos, stats.chi2_contingency(pd.crosstab(ecom["dispositivo"], ecom["grupo"]))[1]), "chi2_contingency(pd.crosstab(dispositivo, grupo))[1]"
_d = (conv.loc["B", "sum"] / conv.loc["B", "count"] - conv.loc["A", "sum"] / conv.loc["A", "count"]) * 100
assert np.isclose(dif_pp, _d), "dif_pp = (tasso di B − tasso di A) · 100"
_lo, _hi = confint_proportions_2indep(conv.loc["B", "sum"], conv.loc["B", "count"], conv.loc["A", "sum"], conv.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf_pp, _lo * 100) and np.isclose(ic_sup_pp, _hi * 100), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb'), moltiplicato per 100"
assert ic_inf_pp < dif_pp < ic_sup_pp, "l'IC deve contenere la stima"
assert np.isclose(p_valor, proportions_ztest([conv.loc["B", "sum"], conv.loc["A", "sum"]], [conv.loc["B", "count"], conv.loc["A", "count"]])[1]), "proportions_ztest([conv_B, conv_A], [n_B, n_A])[1]"
print(f"Utenti: {tamanos_b}  SRM: chi² = {chi2_srm:.1f}, p = {p_srm:.2g}  (allarme!)" if srm_detectado else f"SRM: p = {p_srm:.3f}")
print(f"Dispositivo × gruppo: p = {p_dispositivos:.2g}")
print(f"Differenza B − A = {dif_pp:.2f} pp (IC 95 %: {ic_inf_pp:.2f} a {ic_sup_pp:.2f}), p = {p_valor:.4f}")
print("Caso B, passo 2 corretto ✔")

### Caso B · Passo 3 — Segmenti con correzione e decisione

Per ciascuno dei 6 segmenti `dispositivo × canal` (nell'ordine di `ecom.groupby(["dispositivo", "canal"])`) calcola il p-value del test per due proporzioni (B per primo): `p_seg` (lista di 6 valori). Correggi con **Holm**: `p_holm` (`multipletests(p_seg, method="holm")[1]`). Conta `n_sig_sin` (segmenti con `p_seg < 0.05`) e `n_sig_holm` (con `p_holm < 0.05`). Infine costruisci `resultado_b` con: `n`, `dif_pp`, `ic_inf_pp`, `ic_sup_pp`, `srm_p` (= `p_srm`), `n_seg_sig_holm` (= `n_sig_holm`) e `decision`: `"revisar_asignacion"` se c'è SRM; altrimenti `"lanzar_B"` quando l'IC della differenza non include 0 e `"no_lanzar_B"` in caso contrario.

In [ ]:
p_seg = []
for _, seg in ecom.groupby(["dispositivo", "canal"]):
    pass    # TODO: conversioni e utenti di A e di B nel segmento; aggiungi il p-value a p_seg

p_holm = None       # TODO
n_sig_sin = None    # TODO
n_sig_holm = None   # TODO

resultado_b = {
    "n": None,               # TODO
    "dif_pp": None,          # TODO
    "ic_inf_pp": None,       # TODO
    "ic_sup_pp": None,       # TODO
    "srm_p": None,           # TODO
    "n_seg_sig_holm": None,  # TODO
    "decision": None,        # TODO
}

# Controllo automatico
assert p_holm is not None and n_sig_sin is not None and n_sig_holm is not None and all(v is not None for v in resultado_b.values()), "Completa le righe contrassegnate con TODO"
_ps = []
for _, _s in ecom.groupby(["dispositivo", "canal"]):
    _t = _s.groupby("grupo")["convirtio"].agg(["sum", "count"])
    _ps.append(proportions_ztest([_t.loc["B", "sum"], _t.loc["A", "sum"]], [_t.loc["B", "count"], _t.loc["A", "count"]])[1])
assert len(p_seg) == 6 and np.allclose(p_seg, _ps), "6 p-value, uno per segmento, con B per primo in proportions_ztest"
assert np.allclose(p_holm, multipletests(_ps, method="holm")[1]), "multipletests(p_seg, method='holm')[1]"
assert n_sig_sin == int(np.sum(np.array(_ps) < 0.05)) and n_sig_holm == int(np.sum(multipletests(_ps, method="holm")[1] < 0.05)), "conta quanti p-value (non corretti e corretti) sono < 0.05"
assert n_sig_holm <= n_sig_sin, "correggere non può mai aumentare il numero di significativi"
_r = resultado_b
assert _r["n"] == len(ecom) and np.isclose(_r["dif_pp"], dif_pp) and np.isclose(_r["ic_inf_pp"], ic_inf_pp) and np.isclose(_r["ic_sup_pp"], ic_sup_pp), "usa le cifre del passo 2"
assert np.isclose(_r["srm_p"], p_srm) and _r["n_seg_sig_holm"] == n_sig_holm, "srm_p = p_srm e n_seg_sig_holm = n_sig_holm"
assert _r["ic_inf_pp"] < _r["dif_pp"] < _r["ic_sup_pp"], "l'IC deve contenere la stima"
assert _r["decision"] == ("revisar_asignacion" if p_srm < 0.001 else ("lanzar_B" if ic_inf_pp > 0 else "no_lanzar_B")), "se c'è SRM la decisione è rivedere l'assegnazione, anche se il p-value è piccolo"
print(f"Segmenti significativi senza correzione: {n_sig_sin}; con Holm: {n_sig_holm}")
print(f"Decisione: {_r['decision']}")
print("Caso B, passo 3 corretto ✔")

---
# Caso C — Immobiliare: da cosa dipende il valore dell'abitazione?

**Contesto.** 5 000 distretti censuari della California (i dati della lezione 10, campione casuale con seme 42): `MedInc` (reddito mediano, in decine di migliaia di $), `HouseAge`, `AveRooms` e `AveBedrms` (stanze e camere da letto per abitazione), `Population`, `AveOccup` (occupanti per abitazione), `Latitude`, `Longitude` e `MedHouseVal` (**valore mediano** dell'abitazione in unità da 100 000 $, **censurato** a 5,00001). Sono dati **osservazionali** e aggregati per distretto.

**Domanda di business.** Un'agenzia vuole sapere quanto il reddito del distretto si associa al valore dell'abitazione e quali altre caratteristiche contano.

**Rotta.** Audit con **filtri ragionati** (`AveRooms < 10` e `AveOccup < 6`), censura (lezioni 2-3 e 10) → regressione multipla con **HC3** per l'eteroschedasticità e VIF (lezioni 10 e 12) → effetto di `MedInc` con IC.
**Trappole:** la **censura** a 5,00001; **AveRooms** cambia segno passando dal modello semplice a quello multiplo; l'adattamento non è una previsione (lezione 12); ed è un dato **ecologico** (distretti, non persone).

### Caso C · Passo 1 — Audit, filtri e censura

Su `cali` calcola: `n_bruto` (righe), `n_faltantes_c` (totale dei mancanti), `n_duplicados_c` (righe duplicate) e `n_censurados_bruto` (distretti con `MedHouseVal >= 5`). Poi crea `cali_f`, con `.copy()`, tenendo le righe con `AveRooms < 10` **e** `AveOccup < 6` (valori estremi che quasi certamente sono errori o distretti atipici), e calcola `n_filtrado`, `n_censurados` (censurati che restano in `cali_f`) e `r_rooms` (correlazione di `AveRooms` con `MedHouseVal` in `cali_f`).

In [ ]:
n_bruto = None            # TODO
n_faltantes_c = None      # TODO
n_duplicados_c = None     # TODO
n_censurados_bruto = None # TODO

cali_f = None             # TODO: filtra con la condizione congiunta e usa .copy()
n_filtrado = None         # TODO
n_censurados = None       # TODO
r_rooms = None            # TODO

# Controllo automatico
assert all(v is not None for v in (n_bruto, n_faltantes_c, n_duplicados_c, n_censurados_bruto, cali_f, n_filtrado, n_censurados, r_rooms)), "Completa le righe contrassegnate con TODO"
assert n_bruto == len(cali) and n_faltantes_c == int(cali.isna().sum().sum()) and n_duplicados_c == int(cali.duplicated().sum()), "len(cali), cali.isna().sum().sum(), cali.duplicated().sum()"
assert n_censurados_bruto == int((cali["MedHouseVal"] >= 5).sum()), "distretti con MedHouseVal >= 5"
_f = cali[(cali["AveRooms"] < 10) & (cali["AveOccup"] < 6)]
assert n_filtrado == len(_f) and len(cali_f) == len(_f), "cali[(cali['AveRooms'] < 10) & (cali['AveOccup'] < 6)]: attenzione alle parentesi e a & (non 'and')"
assert n_censurados == int((_f["MedHouseVal"] >= 5).sum()), "censurati che restano in cali_f"
assert np.isclose(r_rooms, _f["AveRooms"].corr(_f["MedHouseVal"])), "correlazione in cali_f, non in cali"
print(f"Righe: {n_bruto} → {n_filtrado} dopo i filtri; censurati: {n_censurados_bruto} → {n_censurados}")
print(f"Correlazione semplice di AveRooms con il valore: {r_rooms:.3f}")
print("Caso C, passo 1 corretto ✔")

### Caso C · Passo 2 — Modello multiplo, errori robusti e collinearità

Con `cali_f` adatta la regressione `MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + AveOccup` in due modi: `modelo_c_ols` (errori classici) e `modelo_c` (errori **HC3**). Calcola `cociente_se` (errore standard HC3 di `MedInc` diviso per quello classico: di quanto era sottostimata l'incertezza?), `vif_max_c` (maggiore VIF delle cinque esplicative, con `vif_de`) e `signo_cambia` (`True` se il coefficiente di `AveRooms` in `modelo_c` ha **segno diverso** da `r_rooms`).

In [ ]:
formula_c = "MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + AveOccup"

modelo_c_ols = None    # TODO
modelo_c = None        # TODO
cociente_se = None     # TODO
vif_max_c = None       # TODO
signo_cambia = None    # TODO

# Controllo automatico
assert all(v is not None for v in (modelo_c_ols, modelo_c, cociente_se, vif_max_c, signo_cambia)), "Completa le righe contrassegnate con TODO"
_o = smf.ols(formula_c, data=cali_f).fit()
assert np.allclose(modelo_c_ols.params, _o.params) and np.allclose(modelo_c_ols.bse, _o.bse), "modelo_c_ols: smf.ols(formula_c, data=cali_f).fit()"
_h = smf.ols(formula_c, data=cali_f).fit(cov_type="HC3")
assert np.allclose(modelo_c.bse, _h.bse), "modelo_c deve usare .fit(cov_type='HC3')"
assert np.isclose(cociente_se, _h.bse["MedInc"] / _o.bse["MedInc"]), "cociente_se = modelo_c.bse['MedInc'] / modelo_c_ols.bse['MedInc']"
_v = np.diag(np.linalg.inv(np.corrcoef(cali_f[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "AveOccup"]].values.T))).max()
assert np.isclose(vif_max_c, _v, rtol=1e-6), "vif_max_c = max(vif_de(cali_f[[...cinque esplicative...]]).values())"
assert signo_cambia == bool(np.sign(_h.params["AveRooms"]) != np.sign(cali_f["AveRooms"].corr(cali_f["MedHouseVal"]))), "confronta np.sign(coefficiente di AveRooms) con np.sign(r_rooms)"
print(f"L'errore standard di MedInc cresce di un fattore {cociente_se:.2f} con HC3; VIF massimo = {vif_max_c:.2f}; AveRooms cambia segno? {signo_cambia}")
print("Caso C, passo 2 corretto ✔")

### Caso C · Passo 3 — Risultato con incertezza

Costruisci `resultado_c` con le chiavi: `n` (osservazioni di `modelo_c`), `coef_medinc` (coefficiente di `MedInc`: variazione attesa del valore, in unità da 100 000 $, per ogni unità di `MedInc`, a parità del resto), `ic_inf` e `ic_sup` (IC 95 % con HC3), `r2` (R² di `modelo_c_ols`), `n_censurados` (quelli di `cali_f`) e `causal` (`True` o `False`: questi dati permettono di affermare che aumentare il reddito di un distretto **causerà** quell'aumento di valore?).

In [ ]:
resultado_c = {
    "n": None,             # TODO
    "coef_medinc": None,   # TODO
    "ic_inf": None,        # TODO
    "ic_sup": None,        # TODO
    "r2": None,            # TODO
    "n_censurados": None,  # TODO
    "causal": None,        # TODO
}

# Controllo automatico
assert all(v is not None for v in resultado_c.values()), "Completa le righe contrassegnate con TODO"
_r = resultado_c
assert _r["n"] == len(cali_f), "n = numero di osservazioni del modello (modelo_c.nobs)"
assert np.isclose(_r["coef_medinc"], modelo_c.params["MedInc"]), "coef_medinc = modelo_c.params['MedInc']"
_ic = modelo_c.conf_int().loc["MedInc"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 % con HC3: modelo_c.conf_int().loc['MedInc']"
assert _r["ic_inf"] < _r["coef_medinc"] < _r["ic_sup"], "l'IC deve contenere la stima"
assert np.isclose(_r["r2"], modelo_c_ols.rsquared) and 0 < _r["r2"] < 1, "r2 = modelo_c_ols.rsquared"
assert _r["n_censurados"] == int((cali_f["MedHouseVal"] >= 5).sum()), "censurati di cali_f"
assert _r["causal"] is False, "dati osservazionali e aggregati: associazione, non causa"
print(f"MedInc: {_r['coef_medinc']:.3f} per unità (IC 95 % HC3: {_r['ic_inf']:.3f} a {_r['ic_sup']:.3f}); R² = {_r['r2']:.3f}; censurati = {_r['n_censurados']}")
print("Caso C, passo 3 corretto ✔")

---
# Sfida finale — Il report di una pagina

Scrivi il report del caso che hai scelto. Assegna a `caso_elegido` la lettera (`"A"`, `"B"` o `"C"`) e a `informe_final` una stringa con **sei sezioni**, in quest'ordine: **Domanda e decisione · Dati · Metodo · Risultati · Limiti · Raccomandazione** (usa questi titoli). Deve citare le **cifre chiave** del tuo `resultado_x` (dimensione campionaria, stima e intervallo), dichiarare i **limiti** e non superare una pagina (circa 3 500 caratteri). Usa il modello del capitolo (sezione 3). Il controllo automatico verifica solo il minimo; la qualità la valuti tu con la rubrica finale.

In [ ]:
caso_elegido = None    # TODO: "A", "B" o "C"
informe_final = None   # TODO: una stringa con le sei sezioni (puoi usare f-string con i tuoi resultado_x)

# Controllo automatico
assert caso_elegido in ("A", "B", "C") and isinstance(informe_final, str), "Completa le righe contrassegnate con TODO"
_res, _claves = {"A": (resultado_a, ("coef_bmi", "ic_inf", "ic_sup")),
                 "B": (resultado_b, ("dif_pp", "ic_inf_pp", "ic_sup_pp")),
                 "C": (resultado_c, ("coef_medinc", "ic_inf", "ic_sup"))}[caso_elegido]
assert all(v is not None for v in _res.values()), "completa prima i tre passi del tuo caso"

def _norm(t):
    return unicodedata.normalize("NFD", t.lower()).encode("ascii", "ignore").decode()

_texto = _norm(informe_final)
_titulos = ["domanda", "dati", "metodo", "risultati", "limiti", "raccomandazione"]
_faltan = [t for t in _titulos if t not in _texto]
assert not _faltan, f"mancano sezioni (usa i titoli del modello): {_faltan}"
assert 600 <= len(informe_final) <= 5000, "il report deve avere tra 600 e 5 000 caratteri (una pagina)"
assert str(_res["n"]) in re.sub(r"[\s.,]", "", informe_final), "cita la dimensione campionaria n"
_numeros = [float(x.replace(",", ".")) for x in re.findall(r"-?\d+(?:[.,]\d+)?", informe_final)]
for _k in _claves:
    _v = _res[_k]
    assert any(abs(x - _v) <= max(0.06, 0.01 * abs(_v)) for x in _numeros), f"cita la cifra di '{_k}' ({_v:.2f}, arrotondata)"
print(f"Report con le sei sezioni, i limiti e le cifre del caso {caso_elegido} citate.")
print("Sfida finale corretta ✔")

---
## Chiusura della pratica e del corso

Hai percorso un'analisi completa, dalla domanda al report. Con questo **il corso termina**. Per proseguire, il passo naturale è un corso di Machine Learning, che riprende la validazione e i modelli delle lezioni 11 e 12 con strumenti di previsione; e la documentazione di `statsmodels` e `scipy` per approfondire ogni metodo.

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un passo. Eseguile in ordine se vuoi confrontarle con il tuo risultato. La soluzione della sfida finale redige un report di esempio per ogni caso (qui viene mostrato quello del caso B).

### Soluzione del caso A · 1

In [ ]:
n_a = len(diab)
n_faltantes_a = int(diab.isna().sum().sum())
n_duplicados_a = int(diab.duplicated().sum())
r_bmi = diab["bmi"].corr(diab["target"])
var_max_corr = diab.drop(columns="target").corrwith(diab["target"]).abs().idxmax()
r_s1_s2 = diab["s1"].corr(diab["s2"])

# Controllo automatico
assert all(v is not None for v in (n_a, n_faltantes_a, n_duplicados_a, r_bmi, var_max_corr, r_s1_s2)), "Completa le righe contrassegnate con TODO"
assert n_a == len(diab), "n_a = len(diab)"
assert n_faltantes_a == int(diab.isna().sum().sum()), "somma di tutti i mancanti: diab.isna().sum().sum()"
assert n_duplicados_a == int(diab.duplicated().sum()), "diab.duplicated().sum()"
assert np.isclose(r_bmi, diab["bmi"].corr(diab["target"])), "r di Pearson tra bmi e target"
assert var_max_corr == diab.drop(columns="target").corrwith(diab["target"]).abs().idxmax(), "usa il valore assoluto della correlazione e idxmax()"
assert np.isclose(r_s1_s2, diab["s1"].corr(diab["s2"])), "r di Pearson tra s1 e s2"
print(f"n = {n_a}, mancanti = {n_faltantes_a}, duplicati = {n_duplicados_a}, r(bmi, target) = {r_bmi:.3f}, maggiore |r|: {var_max_corr}, r(s1, s2) = {r_s1_s2:.3f}")
print("Caso A, passo 1 corretto ✔")

### Soluzione del caso A · 2

In [ ]:
X_a = diab.drop(columns="target").copy()
X_a["sex"] = (X_a["sex"] == 2).astype(float)     # 0/1: solo per poter calcolare il VIF

vif_a = vif_de(X_a)
vif_max_a = max(vif_a.values())
variables_vif_alto = sorted(c for c, v in vif_a.items() if v > 10)
modelo_a = smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit(cov_type="HC3")

# Controllo automatico
assert all(v is not None for v in (vif_a, vif_max_a, variables_vif_alto, modelo_a)), "Completa le righe contrassegnate con TODO"
_Xn = X_a.astype(float).values
_ref = dict(zip(X_a.columns, np.diag(np.linalg.inv(np.corrcoef(_Xn.T)))))      # VIF = diagonale dell'inversa della matrice di correlazione
assert set(vif_a) == set(X_a.columns) and all(np.isclose(vif_a[c], _ref[c], rtol=1e-6) for c in _ref), "vif_a = vif_de(X_a): un VIF per ogni colonna di X_a"
assert np.isclose(vif_max_a, max(_ref.values())), "vif_max_a = max(vif_a.values())"
assert variables_vif_alto == sorted(c for c, v in _ref.items() if v > 10), "sorted([c for c, v in vif_a.items() if v > 10])"
assert getattr(modelo_a, "cov_type", None) == "HC3", "manca .fit(cov_type='HC3')"
assert set(modelo_a.params.index) == {"Intercept", "C(sex)[T.2.0]", "age", "bmi", "bp", "s3", "s5", "s6"}, "formula: target ~ age + C(sex) + bmi + bp + s3 + s5 + s6"
print(f"VIF massimo (modello completo) = {vif_max_a:.1f}; variabili con VIF > 10:", variables_vif_alto)
print("Coefficiente di bmi (HC3):", round(modelo_a.params["bmi"], 3))
print("Caso A, passo 2 corretto ✔")

### Soluzione del caso A · 3

In [ ]:
X_red = X_a[["age", "sex", "bmi", "bp", "s3", "s5", "s6"]]

ic = modelo_a.conf_int().loc["bmi"]
resultado_a = {
    "n": int(modelo_a.nobs),
    "coef_bmi": modelo_a.params["bmi"],
    "ic_inf": ic[0],
    "ic_sup": ic[1],
    "r2_adj": smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit().rsquared_adj,
    "vif_max": max(vif_de(X_red).values()),
    "causal": False,
}

# Controllo automatico
assert all(v is not None for v in resultado_a.values()), "Completa le righe contrassegnate con TODO"
_r = resultado_a
assert _r["n"] == len(diab), "n = numero di osservazioni del modello (modelo_a.nobs)"
assert np.isclose(_r["coef_bmi"], modelo_a.params["bmi"]), "coef_bmi = modelo_a.params['bmi']"
_ic = modelo_a.conf_int().loc["bmi"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 %: modelo_a.conf_int().loc['bmi']"
assert _r["ic_inf"] < _r["coef_bmi"] < _r["ic_sup"], "l'IC deve contenere la stima"
assert np.isclose(_r["r2_adj"], smf.ols("target ~ age + C(sex) + bmi + bp + s3 + s5 + s6", data=diab).fit().rsquared_adj), "R² corretto: .rsquared_adj (quello del modello senza HC3 è lo stesso valore)"
_v = np.diag(np.linalg.inv(np.corrcoef(X_red.astype(float).values.T))).max()
assert np.isclose(_r["vif_max"], _v, rtol=1e-6), "vif_max = max(vif_de(X_red).values())"
assert _r["causal"] is False, "sono dati osservazionali: non si può affermare una causa"
print(f"Effetto di bmi: {_r['coef_bmi']:.2f} per unità (IC 95 % HC3: {_r['ic_inf']:.2f} a {_r['ic_sup']:.2f}); R² corretto = {_r['r2_adj']:.3f}; VIF massimo = {_r['vif_max']:.2f}")
print("Caso A, passo 3 corretto ✔")

### Soluzione del caso B · 1

In [ ]:
n_b = len(ecom)
n_faltantes_b = int(ecom.isna().sum().sum())
n_duplicados_id = int(ecom["id_usuario"].duplicated().sum())
tasa_A = ecom.loc[ecom["grupo"] == "A", "convirtio"].mean() * 100
tasa_B = ecom.loc[ecom["grupo"] == "B", "convirtio"].mean() * 100
asimetria_ingresos = stats.skew(ecom.loc[ecom["convirtio"] == 1, "ingresos"])

# Controllo automatico
assert all(v is not None for v in (n_b, n_faltantes_b, n_duplicados_id, tasa_A, tasa_B, asimetria_ingresos)), "Completa le righe contrassegnate con TODO"
assert n_b == len(ecom), "n_b = len(ecom)"
assert n_faltantes_b == int(ecom.isna().sum().sum()) and n_duplicados_id == int(ecom["id_usuario"].duplicated().sum()), "ecom.isna().sum().sum() e ecom['id_usuario'].duplicated().sum()"
assert np.isclose(tasa_A, ecom.loc[ecom["grupo"] == "A", "convirtio"].mean() * 100), "tasa_A: media di convirtio nel gruppo A, moltiplicata per 100"
assert np.isclose(tasa_B, ecom.loc[ecom["grupo"] == "B", "convirtio"].mean() * 100), "tasa_B: media di convirtio nel gruppo B, moltiplicata per 100"
assert np.isclose(asimetria_ingresos, stats.skew(ecom.loc[ecom["convirtio"] == 1, "ingresos"])), "stats.skew di ingresos filtrando convirtio == 1"
assert asimetria_ingresos > 1, "un'asimmetria > 1 indica coda lunga: meglio la mediana o il bootstrap della media"
print(f"n = {n_b}, mancanti = {n_faltantes_b}, id ripetuti = {n_duplicados_id}, conversione A = {tasa_A:.2f} %, conversione B = {tasa_B:.2f} %, asimmetria degli introiti = {asimetria_ingresos:.2f}")
print("Caso B, passo 1 corretto ✔")

### Soluzione del caso B · 2

In [ ]:
conv = ecom.groupby("grupo")["convirtio"].agg(["sum", "count"])     # conversioni e utenti per gruppo

tamanos_b = {"A": int(conv.loc["A", "count"]), "B": int(conv.loc["B", "count"])}
chi2_srm, p_srm = stats.chisquare([tamanos_b["A"], tamanos_b["B"]])
srm_detectado = bool(p_srm < 0.001)
p_dispositivos = stats.chi2_contingency(pd.crosstab(ecom["dispositivo"], ecom["grupo"]))[1]
dif_pp = (conv.loc["B", "sum"] / conv.loc["B", "count"] - conv.loc["A", "sum"] / conv.loc["A", "count"]) * 100
lo, hi = confint_proportions_2indep(conv.loc["B", "sum"], conv.loc["B", "count"], conv.loc["A", "sum"], conv.loc["A", "count"], method="newcomb")
ic_inf_pp, ic_sup_pp = lo * 100, hi * 100
p_valor = proportions_ztest([conv.loc["B", "sum"], conv.loc["A", "sum"]], [conv.loc["B", "count"], conv.loc["A", "count"]])[1]

# Controllo automatico
assert all(v is not None for v in (tamanos_b, chi2_srm, p_srm, srm_detectado, p_dispositivos, dif_pp, ic_inf_pp, ic_sup_pp, p_valor)), "Completa le righe contrassegnate con TODO"
assert tamanos_b == {"A": int(conv.loc["A", "count"]), "B": int(conv.loc["B", "count"])}, "tamanos_b = {'A': utenti di A, 'B': utenti di B}"
_c2, _p = stats.chisquare([conv.loc["A", "count"], conv.loc["B", "count"]])
assert np.isclose(chi2_srm, _c2) and np.isclose(p_srm, _p), "stats.chisquare([n_A, n_B]) ripartisce per default in parti uguali"
assert srm_detectado == bool(_p < 0.001), "srm_detectado = p_srm < 0.001"
assert np.isclose(p_dispositivos, stats.chi2_contingency(pd.crosstab(ecom["dispositivo"], ecom["grupo"]))[1]), "chi2_contingency(pd.crosstab(dispositivo, grupo))[1]"
_d = (conv.loc["B", "sum"] / conv.loc["B", "count"] - conv.loc["A", "sum"] / conv.loc["A", "count"]) * 100
assert np.isclose(dif_pp, _d), "dif_pp = (tasso di B − tasso di A) · 100"
_lo, _hi = confint_proportions_2indep(conv.loc["B", "sum"], conv.loc["B", "count"], conv.loc["A", "sum"], conv.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf_pp, _lo * 100) and np.isclose(ic_sup_pp, _hi * 100), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb'), moltiplicato per 100"
assert ic_inf_pp < dif_pp < ic_sup_pp, "l'IC deve contenere la stima"
assert np.isclose(p_valor, proportions_ztest([conv.loc["B", "sum"], conv.loc["A", "sum"]], [conv.loc["B", "count"], conv.loc["A", "count"]])[1]), "proportions_ztest([conv_B, conv_A], [n_B, n_A])[1]"
print(f"Utenti: {tamanos_b}  SRM: chi² = {chi2_srm:.1f}, p = {p_srm:.2g}  (allarme!)" if srm_detectado else f"SRM: p = {p_srm:.3f}")
print(f"Dispositivo × gruppo: p = {p_dispositivos:.2g}")
print(f"Differenza B − A = {dif_pp:.2f} pp (IC 95 %: {ic_inf_pp:.2f} a {ic_sup_pp:.2f}), p = {p_valor:.4f}")
print("Caso B, passo 2 corretto ✔")

### Soluzione del caso B · 3

In [ ]:
p_seg = []
for _, seg in ecom.groupby(["dispositivo", "canal"]):
    t = seg.groupby("grupo")["convirtio"].agg(["sum", "count"])
    p_seg.append(proportions_ztest([t.loc["B", "sum"], t.loc["A", "sum"]], [t.loc["B", "count"], t.loc["A", "count"]])[1])

p_holm = multipletests(p_seg, method="holm")[1]
n_sig_sin = int(np.sum(np.array(p_seg) < 0.05))
n_sig_holm = int(np.sum(p_holm < 0.05))

if srm_detectado:
    decision_b = "revisar_asignacion"
elif ic_inf_pp > 0:
    decision_b = "lanzar_B"
else:
    decision_b = "no_lanzar_B"

resultado_b = {
    "n": n_b,
    "dif_pp": dif_pp,
    "ic_inf_pp": ic_inf_pp,
    "ic_sup_pp": ic_sup_pp,
    "srm_p": p_srm,
    "n_seg_sig_holm": n_sig_holm,
    "decision": decision_b,
}

# Controllo automatico
assert p_holm is not None and n_sig_sin is not None and n_sig_holm is not None and all(v is not None for v in resultado_b.values()), "Completa le righe contrassegnate con TODO"
_ps = []
for _, _s in ecom.groupby(["dispositivo", "canal"]):
    _t = _s.groupby("grupo")["convirtio"].agg(["sum", "count"])
    _ps.append(proportions_ztest([_t.loc["B", "sum"], _t.loc["A", "sum"]], [_t.loc["B", "count"], _t.loc["A", "count"]])[1])
assert len(p_seg) == 6 and np.allclose(p_seg, _ps), "6 p-value, uno per segmento, con B per primo in proportions_ztest"
assert np.allclose(p_holm, multipletests(_ps, method="holm")[1]), "multipletests(p_seg, method='holm')[1]"
assert n_sig_sin == int(np.sum(np.array(_ps) < 0.05)) and n_sig_holm == int(np.sum(multipletests(_ps, method="holm")[1] < 0.05)), "conta quanti p-value (non corretti e corretti) sono < 0.05"
assert n_sig_holm <= n_sig_sin, "correggere non può mai aumentare il numero di significativi"
_r = resultado_b
assert _r["n"] == len(ecom) and np.isclose(_r["dif_pp"], dif_pp) and np.isclose(_r["ic_inf_pp"], ic_inf_pp) and np.isclose(_r["ic_sup_pp"], ic_sup_pp), "usa le cifre del passo 2"
assert np.isclose(_r["srm_p"], p_srm) and _r["n_seg_sig_holm"] == n_sig_holm, "srm_p = p_srm e n_seg_sig_holm = n_sig_holm"
assert _r["ic_inf_pp"] < _r["dif_pp"] < _r["ic_sup_pp"], "l'IC deve contenere la stima"
assert _r["decision"] == ("revisar_asignacion" if p_srm < 0.001 else ("lanzar_B" if ic_inf_pp > 0 else "no_lanzar_B")), "se c'è SRM la decisione è rivedere l'assegnazione, anche se il p-value è piccolo"
print(f"Segmenti significativi senza correzione: {n_sig_sin}; con Holm: {n_sig_holm}")
print(f"Decisione: {_r['decision']}")
print("Caso B, passo 3 corretto ✔")

### Soluzione del caso C · 1

In [ ]:
n_bruto = len(cali)
n_faltantes_c = int(cali.isna().sum().sum())
n_duplicados_c = int(cali.duplicated().sum())
n_censurados_bruto = int((cali["MedHouseVal"] >= 5).sum())

cali_f = cali[(cali["AveRooms"] < 10) & (cali["AveOccup"] < 6)].copy()
n_filtrado = len(cali_f)
n_censurados = int((cali_f["MedHouseVal"] >= 5).sum())
r_rooms = cali_f["AveRooms"].corr(cali_f["MedHouseVal"])

# Controllo automatico
assert all(v is not None for v in (n_bruto, n_faltantes_c, n_duplicados_c, n_censurados_bruto, cali_f, n_filtrado, n_censurados, r_rooms)), "Completa le righe contrassegnate con TODO"
assert n_bruto == len(cali) and n_faltantes_c == int(cali.isna().sum().sum()) and n_duplicados_c == int(cali.duplicated().sum()), "len(cali), cali.isna().sum().sum(), cali.duplicated().sum()"
assert n_censurados_bruto == int((cali["MedHouseVal"] >= 5).sum()), "distretti con MedHouseVal >= 5"
_f = cali[(cali["AveRooms"] < 10) & (cali["AveOccup"] < 6)]
assert n_filtrado == len(_f) and len(cali_f) == len(_f), "cali[(cali['AveRooms'] < 10) & (cali['AveOccup'] < 6)]: attenzione alle parentesi e a & (non 'and')"
assert n_censurados == int((_f["MedHouseVal"] >= 5).sum()), "censurati che restano in cali_f"
assert np.isclose(r_rooms, _f["AveRooms"].corr(_f["MedHouseVal"])), "correlazione in cali_f, non in cali"
print(f"Righe: {n_bruto} → {n_filtrado} dopo i filtri; censurati: {n_censurados_bruto} → {n_censurados}")
print(f"Correlazione semplice di AveRooms con il valore: {r_rooms:.3f}")
print("Caso C, passo 1 corretto ✔")

### Soluzione del caso C · 2

In [ ]:
formula_c = "MedHouseVal ~ MedInc + HouseAge + AveRooms + AveBedrms + AveOccup"

modelo_c_ols = smf.ols(formula_c, data=cali_f).fit()
modelo_c = smf.ols(formula_c, data=cali_f).fit(cov_type="HC3")
cociente_se = modelo_c.bse["MedInc"] / modelo_c_ols.bse["MedInc"]
vif_max_c = max(vif_de(cali_f[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "AveOccup"]]).values())
signo_cambia = bool(np.sign(modelo_c.params["AveRooms"]) != np.sign(r_rooms))

# Controllo automatico
assert all(v is not None for v in (modelo_c_ols, modelo_c, cociente_se, vif_max_c, signo_cambia)), "Completa le righe contrassegnate con TODO"
_o = smf.ols(formula_c, data=cali_f).fit()
assert np.allclose(modelo_c_ols.params, _o.params) and np.allclose(modelo_c_ols.bse, _o.bse), "modelo_c_ols: smf.ols(formula_c, data=cali_f).fit()"
_h = smf.ols(formula_c, data=cali_f).fit(cov_type="HC3")
assert np.allclose(modelo_c.bse, _h.bse), "modelo_c deve usare .fit(cov_type='HC3')"
assert np.isclose(cociente_se, _h.bse["MedInc"] / _o.bse["MedInc"]), "cociente_se = modelo_c.bse['MedInc'] / modelo_c_ols.bse['MedInc']"
_v = np.diag(np.linalg.inv(np.corrcoef(cali_f[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "AveOccup"]].values.T))).max()
assert np.isclose(vif_max_c, _v, rtol=1e-6), "vif_max_c = max(vif_de(cali_f[[...cinque esplicative...]]).values())"
assert signo_cambia == bool(np.sign(_h.params["AveRooms"]) != np.sign(cali_f["AveRooms"].corr(cali_f["MedHouseVal"]))), "confronta np.sign(coefficiente di AveRooms) con np.sign(r_rooms)"
print(f"L'errore standard di MedInc cresce di un fattore {cociente_se:.2f} con HC3; VIF massimo = {vif_max_c:.2f}; AveRooms cambia segno? {signo_cambia}")
print("Caso C, passo 2 corretto ✔")

### Soluzione del caso C · 3

In [ ]:
ic = modelo_c.conf_int().loc["MedInc"]
resultado_c = {
    "n": int(modelo_c.nobs),
    "coef_medinc": modelo_c.params["MedInc"],
    "ic_inf": ic[0],
    "ic_sup": ic[1],
    "r2": modelo_c_ols.rsquared,
    "n_censurados": n_censurados,
    "causal": False,
}

# Controllo automatico
assert all(v is not None for v in resultado_c.values()), "Completa le righe contrassegnate con TODO"
_r = resultado_c
assert _r["n"] == len(cali_f), "n = numero di osservazioni del modello (modelo_c.nobs)"
assert np.isclose(_r["coef_medinc"], modelo_c.params["MedInc"]), "coef_medinc = modelo_c.params['MedInc']"
_ic = modelo_c.conf_int().loc["MedInc"]
assert np.isclose(_r["ic_inf"], _ic[0]) and np.isclose(_r["ic_sup"], _ic[1]), "IC 95 % con HC3: modelo_c.conf_int().loc['MedInc']"
assert _r["ic_inf"] < _r["coef_medinc"] < _r["ic_sup"], "l'IC deve contenere la stima"
assert np.isclose(_r["r2"], modelo_c_ols.rsquared) and 0 < _r["r2"] < 1, "r2 = modelo_c_ols.rsquared"
assert _r["n_censurados"] == int((cali_f["MedHouseVal"] >= 5).sum()), "censurati di cali_f"
assert _r["causal"] is False, "dati osservazionali e aggregati: associazione, non causa"
print(f"MedInc: {_r['coef_medinc']:.3f} per unità (IC 95 % HC3: {_r['ic_inf']:.3f} a {_r['ic_sup']:.3f}); R² = {_r['r2']:.3f}; censurati = {_r['n_censurados']}")
print("Caso C, passo 3 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def num(x, d=2):
    return f"{x:.{d}f}".replace(".", ",")

textos = {
    "A": f"""### Report — Progressione del diabete

**Domanda e decisione.** Quali caratteristiche di base si associano a una maggiore progressione della malattia? Decisione: chi mettere in priorità nel follow-up.

**Dati.** {resultado_a["n"]} pazienti, senza mancanti né duplicati; dati osservazionali. `s1`, `s2` e `s3` sono quasi ridondanti (VIF fino a {num(vif_max_a, 0)}).

**Metodo.** Regressione lineare multipla senza le variabili ridondanti (VIF massimo {num(resultado_a["vif_max"])} nel modello finale) e errori standard robusti HC3.

**Risultati.** Per ogni unità in più di `bmi` la progressione attesa sale di {num(resultado_a["coef_bmi"])} punti (IC 95 %: {num(resultado_a["ic_inf"])} a {num(resultado_a["ic_sup"])}), a parità delle altre variabili. Il modello spiega circa la metà della variazione (R² corretto = {num(resultado_a["r2_adj"], 3)}).

**Limiti.** Dati osservazionali di {resultado_a["n"]} pazienti: è un'associazione, non una causa. La scelta delle variabili è stata fatta con gli stessi dati (gli IC sono un po' ottimisti) e non è stata validata su dati nuovi. Il segno di `sex` non si interpreta perché la sua codifica non è documentata.

**Raccomandazione.** Usare il `bmi` e la pressione come segnali per dare priorità al follow-up, e validare il modello su dati nuovi prima di usarlo per prevedere.
""",
    "B": f"""### Report — Test A/B della nuova pagina di pagamento

**Domanda e decisione.** La versione B aumenta la conversione? Decisione: lanciarla a tutti gli utenti o no.

**Dati.** {resultado_b["n"]} utenti (sintetici), senza mancanti né id ripetuti. Ripartizione A/B: {tamanos_b["A"]} / {tamanos_b["B"]}.

**Metodo.** Controllo SRM (chi-quadro), equilibrio dei dispositivi, test per due proporzioni con IC della differenza e sei segmenti dispositivo × canale con correzione di Holm.

**Risultati.** Conversione A = {num(tasa_A)} % e B = {num(tasa_B)} %: differenza di {num(resultado_b["dif_pp"])} pp (IC 95 %: {num(resultado_b["ic_inf_pp"])} a {num(resultado_b["ic_sup_pp"])}). Senza correzione, {n_sig_sin} segmenti risultano significativi; con Holm, {resultado_b["n_seg_sig_holm"]}. Ma la ripartizione ha **SRM** (p = {p_srm:.1e}) e i dispositivi non sono equilibrati tra i gruppi (p = {p_dispositivos:.3f}).

**Limiti.** Con SRM, la randomizzazione non è affidabile e la differenza osservata può essere distorta (più mobile in B, dove l'effetto è diverso). I segmenti hanno poca potenza. Gli introiti, con coda lunga, non sono stati analizzati con le medie.

**Raccomandazione.** Non decidere ancora: indagare la causa dell'SRM, correggerla e ripetere o rianalizzare (stratificando per dispositivo). Se si conferma, l'effetto plausibile è positivo ma modesto.
""",
    "C": f"""### Report — Valore delle abitazioni in California

**Domanda e decisione.** Quanto si associa il reddito del distretto al valore dell'abitazione? Decisione: quali variabili usare per orientare la valutazione delle zone.

**Dati.** {n_bruto} distretti; si escludono {n_bruto - n_filtrado} con `AveRooms` ≥ 10 o `AveOccup` ≥ 6 (valori estremi). Ne restano {resultado_c["n"]}, di cui {resultado_c["n_censurados"]} hanno il valore censurato a 5,00001. Nessun mancante né duplicato.

**Metodo.** Regressione lineare multipla con errori robusti HC3 (i classici sottostimavano l'incertezza di un fattore {num(cociente_se)}); VIF massimo {num(vif_max_c)}.

**Risultati.** Per ogni unità in più di reddito mediano (10 000 $), il valore mediano atteso sale di {num(resultado_c["coef_medinc"], 3)} (unità da 100 000 $; IC 95 %: {num(resultado_c["ic_inf"], 3)} a {num(resultado_c["ic_sup"], 3)}), a parità del resto. R² = {num(resultado_c["r2"], 3)}. `AveRooms` cambia segno tra il modello semplice (r = {num(r_rooms)}) e quello multiplo.

**Limiti.** Dati osservazionali e aggregati per distretto (fallacia ecologica): associazione, non causa. La censura a 5,00001 sottostima i distretti più cari; non sono state usate latitudine e longitudine, quindi manca l'effetto della posizione.

**Raccomandazione.** Usare il reddito come primo segnale di valore, con cautela nei distretti di valore molto alto, e incorporare la posizione prima di prendere decisioni per zona.
""",
}
caso_elegido = "B"
informe_final = textos[caso_elegido]
print(informe_final)

assert caso_elegido in ("A", "B", "C") and isinstance(informe_final, str), "Completa le righe contrassegnate con TODO"
_res, _claves = {"A": (resultado_a, ("coef_bmi", "ic_inf", "ic_sup")),
                 "B": (resultado_b, ("dif_pp", "ic_inf_pp", "ic_sup_pp")),
                 "C": (resultado_c, ("coef_medinc", "ic_inf", "ic_sup"))}[caso_elegido]
assert all(v is not None for v in _res.values()), "completa prima i tre passi del tuo caso"

def _norm(t):
    return unicodedata.normalize("NFD", t.lower()).encode("ascii", "ignore").decode()

_texto = _norm(informe_final)
_titulos = ["domanda", "dati", "metodo", "risultati", "limiti", "raccomandazione"]
_faltan = [t for t in _titulos if t not in _texto]
assert not _faltan, f"mancano sezioni (usa i titoli del modello): {_faltan}"
assert 600 <= len(informe_final) <= 5000, "il report deve avere tra 600 e 5 000 caratteri (una pagina)"
assert str(_res["n"]) in re.sub(r"[\s.,]", "", informe_final), "cita la dimensione campionaria n"
_numeros = [float(x.replace(",", ".")) for x in re.findall(r"-?\d+(?:[.,]\d+)?", informe_final)]
for _k in _claves:
    _v = _res[_k]
    assert any(abs(x - _v) <= max(0.06, 0.01 * abs(_v)) for x in _numeros), f"cita la cifra di '{_k}' ({_v:.2f}, arrotondata)"
print(f"Report con le sei sezioni, i limiti e le cifre del caso {caso_elegido} citate.")
print("Sfida finale corretta ✔")

---
## Autovalutazione con la rubrica

Assegna al tuo progetto un punteggio da 0 (non fatto) a 2 (fatto bene) per ogni criterio e annota il totale (massimo 20):

| # | Criterio | 0 / 1 / 2 |
|---|---|---|
| 1 | Domanda di business e decisione definite | |
| 2 | Audit dei dati (tipi, mancanti, duplicati, intervalli) | |
| 3 | Esplorazione prima di modellare | |
| 4 | Metodo giustificato (tabella decisionale) | |
| 5 | Supposti e diagnostica verificati | |
| 6 | Effetto + intervallo di confidenza, non solo p-value | |
| 7 | Molteplicità, selezione delle variabili e collinearità controllate | |
| 8 | Limiti e distorsioni dichiarati | |
| 9 | Nessuna affermazione causale indebita | |
| 10 | Report chiaro, breve e riproducibile | |

Poi annota:

- **Quale caso ho scelto e quale passo mi è costato di più?**
- **Quale strumento di quale lezione dovrei ripassare?**
- **Tempo dedicato:** ___ minuti
- **Cosa farei diversamente con più tempo o più dati:** ...